# 08 - FINAL-TOKEN REPAIR: one consolidated GPU session

Repairs audit RED-1: the committed causal pipeline built the A-D direction from
`*_pooled.npy` (mean of the last 5 tokens); `analysis_plan.md` section 4 fixes it on the
**final non-padding prompt token** (`*_final.npy`). This notebook regenerates the causal
endpoints with `--pooling final_token`.

**Runtime -> Run all**, then leave it. Expect **3-5 h** on a T4.

| job | what | GPU | ~time |
|---|---|---|---|
| pre | fetch Drive artifacts, verify SHA256, ensure 654-row activations for all 4 DPO branches | - | ~10 min |
| 0 | **M3 held-out SMOKE TEST** (final-token) - STOP if it fails | yes | ~10 min |
| A | held-out final-token causal, 4 branches (M3, M3_alt, M3_direct, M3_direct_alt) | yes | ~1.5 h |
| B | 5-fold cross-fitted final-token causal, 4 branches | yes | ~40 min |
| C | full-A/D final-token sensitivity, 4 branches *(only if the paper keeps that table)* | yes | ~1.5 h |
| D | judge (StrongREJECT + WildGuard) every new final-token output | yes | ~1.5-2.5 h |
| post | CPU: final-token confirmatory endpoints + McNemar + comparison; archive to Drive | - | ~5 min |

**Steering is NOT run**: the manuscript reports no steering result or figure (only a
one-line procedure mention in sec:design). See task section 8.E.

**Isolation guarantees** (nothing here can touch the pooled results):
- final-token directions -> `results/refusal_direction/{stage}_v2_direction_final_token.npy`
- final-token causal raw -> `results/raw/causal_ablation_v2_{stage}_L24-28_finaltoken*.json`
- final-token condition names -> `{stage}_ft_baseline` / `{stage}_ft_ablated_AD` / `{stage}_ft_xfit_*`
- the committed pooled `*_v2_direction.npy`, `causal_ablation_v2_*` (no `_finaltoken`) and
  `confirmatory_endpoints.json` are read-only here.


## 1. Pin the repository commit and clone


In [ ]:
PINNED_COMMIT = "abe10c956b31b79d5fcb02cad89ee2a181efebca"
import os, subprocess, sys
REPO = "https://github.com/<owner>/dpo-safety-representations.git"  # TODO: set your remote
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
print("HEAD:", subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())
assert subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip() == PINNED_COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 2. Mount Drive + auto-locate the real `dpo_v2` folder


In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import glob, os
cands = ["/content/drive/MyDrive/dpo_v2"]
cands += sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2"))
cands += sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2"))
REAL = next((c for c in cands if glob.glob(os.path.join(c,"results","activations","*_final.npy"))), None)
assert REAL, "No dpo_v2 folder with real activations found:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL; print("DPO_DRIVE_ROOT =", REAL)
from src.colab_persist import bind, status_line
info = bind(persist_hf_cache=False)   # judges (~20 GB) stay OFF Drive
print(status_line(info))


## 3. HuggingFace auth (needed for job D only)
The account must have accepted **allenai/wildguard** (and google/gemma-2b for the SR lineage).


In [ ]:
from google.colab import userdata
from huggingface_hub import login
_tok = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = _tok
login(token=_tok); print("HF login OK")


## 4. Fetch + SHA256-verify every required input
Fails fast with an exact missing-artifact list. Ensures a **654-row** `_final.npy` +
metadata exists for **all four** DPO branches - M3_direct / M3_direct_alt are 370-row in
the public repo and must come from the Drive `activations_bundle`.


In [ ]:
import json, hashlib, shutil, numpy as np
from pathlib import Path
FROZEN_BENCH = "e4946b070f441c7a0676db830c65257b78a2d1b46abb0a61cce4cc86352f838b"
BRANCHES = ["M3", "M3_alt", "M3_direct", "M3_direct_alt"]
CF3_STAGES = ["M2", "M3"]

def sha(p):
    h=hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()

def pull(rel):
    src = Path(REAL) / rel; dst = Path(rel)
    if src.exists() and (not dst.exists() or src.stat().st_size != dst.stat().st_size):
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    return dst.exists()

missing = []
# benchmark + split manifest
for rel in ["data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl",
            "data/frozen_v2/LATEST_BENCHMARK.json", "logs/direction_split_manifest.json"]:
    if not (Path(rel).exists() or pull("results/../" + rel) or pull(rel)):
        missing.append(rel)
assert sha("data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl") == FROZEN_BENCH, "benchmark hash mismatch"

# activations: need 654-row _final.npy + _pooled.npy + metadata for M2, M3, and all 4 branches
for st in sorted(set(BRANCHES) | set(CF3_STAGES) | {"M2_alt","M3_alt"}):
    for kind in ("final", "pooled"):
        rel = f"results/activations/{st}_{kind}.npy"
        pull(rel)
        if not Path(rel).exists():
            missing.append(rel); continue
        n = np.load(rel, mmap_mode="r").shape[0]
        if n != 654:
            missing.append(f"{rel} (has {n} rows, need 654 - fetch the fresh bundle or re-extract)")
    for rel in (f"results/activations/{st}_metadata.json",
                f"results/activations/{st}_metadata_binding.json"):
        pull(rel)
        if not Path(rel).exists(): missing.append(rel)

if missing:
    print("MISSING / WRONG-SHAPE ARTIFACTS:\n  " + "\n  ".join(missing))
    raise SystemExit("Fix the inputs above before running any GPU job. Do NOT start a partial run.")
print("all required inputs present and SHA-verified")


## 4b. If any branch still lacks 654-row activations: extract them (GPU)
Run only if cell 4 reported a `(has 370 rows ...)` line and the Drive bundle did not supply it.


In [ ]:
# for st in ["M3_direct", "M3_direct_alt"]:
#     subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "extract",
#                     "--stages", st], check=True)
print("skip unless cell 4 flagged a 370-row branch")


## 5. Build the final-token directions (CPU, torch-free) + fold directions


In [ ]:
r = subprocess.run([sys.executable, "-m", "src.analysis.final_token_repair",
                    "--stages", "M2", "M3", "M2_alt", "M3_alt", *[b for b in ["M3_direct","M3_direct_alt"] if Path(f"results/activations/{b}_final.npy").exists() and np.load(f"results/activations/{b}_final.npy",mmap_mode="r").shape[0]==654],
                    "--recompute-cf3",
                    "--out-dir", "results/final_token_repair/summaries"], check=True)
ftdir = json.load(open("results/final_token_repair/summaries/final_token_directions.json"))
for st, e in ftdir["stages"].items():
    vp = e.get("vs_committed_pooled_v2_direction") or {}
    c = (vp.get("cos_final_token_vs_pooled_v2_per_layer") or [None]*25)[24]
    print(f"  {st}: final-token dir built; cos(final, committed pooled v2)@L24 = {c}")
cf3 = json.load(open("results/final_token_repair/summaries/final_token_cf3.json"))
print("\nCF3 (final-token):", cf3["M2"]["macro_f1"], cf3["M3"]["macro_f1"],
      cf3["cf3_macroF1_M3_minus_M2"], cf3["bootstrap_group_diff"]["ci_low"], cf3["bootstrap_group_diff"]["ci_high"])
assert cf3["bootstrap_group_diff"]["ci_low"] < 0 < cf3["bootstrap_group_diff"]["ci_high"], "CF3 CI should span zero"


## JOB 0 - M3 held-out SMOKE TEST (final-token). STOP if it fails.
One stage, the three required conditions, on the held-out A/D + all B/C rows. Verifies the
final-token direction loads, the hooks fire at L24-28, condition names are `M3_ft_*`, the
binding records `pooling=final_token`, and the raw JSON schema is intact.


In [ ]:
import time; t0 = time.time()
subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                "--stage", "M3", "--pooling", "final_token"], check=True)
raw = json.load(open("results/raw/causal_ablation_v2_M3_L24-28_finaltoken.json"))
bind = json.load(open("results/raw/causal_ablation_v2_M3_L24-28_finaltoken_binding.json"))
conds = sorted({r["stage"] for r in raw})
print("conditions:", conds, " rows:", len(raw), " binding.pooling:", bind.get("pooling"))
assert conds == ["M3_ft_ablated_AD", "M3_ft_ablated_random", "M3_ft_baseline"], conds
assert bind["pooling"] == "final_token" and bind["pool_window"] is None
assert bind["benchmark_sha256"] == FROZEN_BENCH
assert len(raw) == 3 * 414, f"expected 3x414 rows (held-out A/D + all B/C), got {len(raw)}"
assert all(r.get("response") for r in raw[:20]), "empty responses"
# the pooled M3 file must be untouched
assert Path("results/raw/causal_ablation_v2_M3_L24-28.json").exists()
pooled_conds = sorted({r["stage"] for r in json.load(open("results/raw/causal_ablation_v2_M3_L24-28.json"))})
assert pooled_conds == ["M3_ablated_AD", "M3_ablated_random", "M3_baseline"], "pooled file changed!"
print(f"SMOKE TEST PASSED in {time.time()-t0:.0f}s -- proceeding to jobs A/B/C/D")


## JOB A - held-out final-token causal, all 4 branches


In [ ]:
for br in BRANCHES:
    if br == "M3":  # already done by the smoke test
        continue
    subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                    "--stage", br, "--pooling", "final_token"], check=True)
for br in BRANCHES:
    p = f"results/raw/causal_ablation_v2_{br}_L24-28_finaltoken.json"
    raw = json.load(open(p)); conds = sorted({r["stage"] for r in raw})
    assert conds == [f"{br}_ft_ablated_AD", f"{br}_ft_ablated_random", f"{br}_ft_baseline"], (br, conds)
    assert len(raw) == 3 * 414, (br, len(raw))
    print(f"  {br}: held-out final-token OK ({len(raw)} rows)")


## JOB B - 5-fold cross-fitted final-token causal, all 4 branches


In [ ]:
for br in BRANCHES:
    subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                    "--stage", br, "--cross-fit", "5", "--pooling", "final_token"], check=True)
for br in BRANCHES:
    p = f"results/raw/causal_ablation_v2_{br}_L24-28_xfit5_finaltoken.json"
    b = json.load(open(p.replace(".json", "_binding.json")))
    raw = json.load(open(p)); conds = sorted({r["stage"] for r in raw})
    assert conds == [f"{br}_ft_xfit_ablated_AD", f"{br}_ft_xfit_ablated_random", f"{br}_ft_xfit_baseline"], (br, conds)
    assert b["cross_fit_k"] == 5 and b["pooling"] == "final_token"
    assert b["n_test_rows_total"] == 120 and len(raw) == 3 * 120, (br, b["n_test_rows_total"], len(raw))
    # fold partition must match the committed pooled xfit5 (prompt-paired)
    comm = {f["fold"]: sorted(f["test_record_ids"]) for f in
            json.load(open(f"results/raw/causal_ablation_v2_{br}_L24-28_xfit5_binding.json"))["folds"]}
    got = {f["fold"]: sorted(f["test_record_ids"]) for f in b["folds"]}
    assert got == comm, f"{br}: final-token fold partition != committed pooled partition"
    print(f"  {br}: cross-fit final-token OK, partition matches committed pooled")


## JOB C - full-A/D final-token sensitivity, all 4 branches
**Run only if the paper keeps the full-A/D table.** If it is dropped, leave this cell
unrun and record that decision in the report - do NOT relabel the pooled full-A results.


In [ ]:
RUN_FULL_AD = True   # set False if the paper drops the full-A/D sensitivity table
if RUN_FULL_AD:
    for br in BRANCHES:
        subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                        "--stage", br, "--all-ad-sensitivity", "--pooling", "final_token"], check=True)
    for br in BRANCHES:
        p = f"results/raw/causal_ablation_v2_{br}_L24-28_finaltoken_fullAD.json"
        raw = json.load(open(p)); conds = sorted({r["stage"] for r in raw})
        assert conds == [f"{br}_ft_ablated_AD", f"{br}_ft_ablated_random", f"{br}_ft_baseline"], (br, conds)
        assert len(raw) == 3 * 300, (br, len(raw))   # 150 A + 150 D
        print(f"  {br}: full-A/D final-token OK ({len(raw)} rows)")
else:
    print("full-A/D final-token SKIPPED by decision - document in the report")


## JOB D - judge (StrongREJECT + WildGuard) every new final-token output
Builds a **final-token-only** consolidated manifest (never mixes in pooled files), then
judges. WildGuard is scored too - it is used for the M3 held-out secondary endpoint.


In [ ]:
import glob
ft_files = sorted(glob.glob("results/raw/causal_ablation_v2_*_L24-28_finaltoken*.json"))
ft_files = [f for f in ft_files if not f.endswith("_binding.json")]
entries = [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")} for f in ft_files]
man = {"kind": "consolidated_response_manifest", "pooling": "final_token",
       "benchmark_sha256": FROZEN_BENCH,
       "split_manifest_sha256": "880381606de7aa2ffbdb8f7c75303cf4937167ed1a2e1b417afeb33761fcf8f1",
       "entries": entries}
Path("results/final_token_repair/manifests").mkdir(parents=True, exist_ok=True)
mp = "results/final_token_repair/manifests/consolidated_judge_final_token.json"
json.dump(man, open(mp, "w"), indent=2)
print(len(entries), "final-token response files ->", mp)
assert all("_finaltoken" in e["response_file"] for e in entries), "non-final-token file leaked into the manifest"

OUT = "results/final_token_repair/judges"
subprocess.run([sys.executable, "-m", "src.analysis.behavioral_judges",
                "--response-manifest", mp, "--run-live", "--require-binding",
                "--reject-legacy", "--out-dir", OUT], check=True)
jf = sorted(glob.glob(OUT + "/behavioral_judges_v2_*.json"))[-1]
print("judged file:", jf)


## POST (CPU) - final-token confirmatory endpoints + McNemar + comparison


In [ ]:
SUM = "results/final_token_repair/summaries"
subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                "--judged", jf, "--condition-infix", "ft_",
                "--out", f"{SUM}/final_token_endpoints.json"], check=True)
e = json.load(open(f"{SUM}/final_token_endpoints.json"))
print("pooling:", e.get("pooling"), " condition_infix:", e.get("condition_infix"))
for st, blk in e["CF2_by_stage"].items():
    for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
        b = blk.get(pop, {})
        print(f"  {st:14s} {pop:20s} n={b.get('n_effective_triples')!s:>4} cf2={b.get('cf2')} CI=[{b.get('ci_low')}, {b.get('ci_high')}]")
xc = e.get("CF2_crossfit_branch_contrasts", {})
print("2x2 interaction:", xc.get("factorial_2x2", {}).get("corpus_x_history_interaction"))

# pooled vs final-token comparison table
pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))
rows = []
for st in e["CF2_by_stage"]:
    for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
        pv = pooled["CF2_by_stage"][st].get(pop, {}).get("cf2")
        fv = e["CF2_by_stage"][st].get(pop, {}).get("cf2")
        if pv is None or fv is None: continue
        rows.append({"stage": st, "population": pop, "pooled_cf2": pv, "final_token_cf2": fv,
                     "abs_diff": abs(pv - fv),
                     "both_CI_exclude_zero": None})  # fill from CI signs below
json.dump({"rows": rows}, open(f"{SUM}/pooled_vs_final_token_CF2.json", "w"), indent=2)
print("wrote", f"{SUM}/pooled_vs_final_token_CF2.json")


## ARCHIVE everything new to Drive + SHA256 manifest


In [ ]:
import tarfile, datetime, hashlib
ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
pay = []
for pat in ["results/raw/causal_ablation_v2_*_finaltoken*.json",
            "results/refusal_direction/*_final_token*.npy",
            "results/refusal_direction/*_final_token*binding.json",
            "results/final_token_repair/**/*"]:
    pay += [p for p in glob.glob(pat, recursive=True) if Path(p).is_file()]
pay = sorted(set(pay))
arch_dir = Path(REAL) / "final_token_repair_archives"; arch_dir.mkdir(exist_ok=True)
tarp = arch_dir / f"final_token_repair_{ts}.tar.gz"
with tarfile.open(tarp, "w:gz") as t:
    for p in pay: t.add(p)
man = {"created_utc": ts, "pinned_commit": PINNED_COMMIT, "n_files": len(pay),
       "archive": str(tarp), "archive_sha256": hashlib.sha256(tarp.read_bytes()).hexdigest(),
       "files": {p: hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in pay}}
json.dump(man, open(arch_dir / f"final_token_repair_{ts}.sha256.json", "w"), indent=2)
json.dump(man, open(f"results/final_token_repair/archive_manifest_{ts}.json", "w"), indent=2)
print("archive:", tarp, "\nsha256:", man["archive_sha256"])


## FINAL - files to download for local reproduction


In [ ]:
print("Download these into the local repo (same paths):\n")
for p in pay:
    print("  ", p)
print("\nThen locally:")
print("  python -m src.analysis.confirmatory_behavioral_endpoints --judged <final-token judged file> \\")
print("      --condition-infix ft_ --out results/final_token_repair/summaries/final_token_endpoints.json")
print("  python -m src.analysis.final_token_summaries --stages M2 M3 M2_alt M3_alt")
print("\nArchive on Drive:", tarp)
